# Pawnee National Grassland Land Swap
## Land Swap Algorithm 
### Compositing contiguous, ecological, economic, oil and gas, and connectivity metrics into a parcel matrix
- **Objective:** In this notebook, we create a land swap algorithm using parcel data and the interior edge ratio metrics calculated in `07_contiguous_area.ipynb` to identify land swaps that would increase the interior edge ratio value. 

A future objective of this notebook is to incorporate the spatial metrics for species, land value, connectivity value, and oil and gas into consideration for the land swaps. The matrix should consider the total value from the metrics to identify best swaps. 

- **Author:** Max Warnock
- **Code review and/or edits:** 
- **Date:** June 17, 2026
- **Last date of revision:** June 26, 2026

---
### 🛠️ Prerequisites & Setup
* **Libraries:** 
* **Environment:**
* **Data Sources:** 
* **Related Notebooks:** 
* **Notes:**

### 🏗️ Methodology
1. Classify parcel ownership (federal, state, private) and project all parcel data to EPSG:5070 equal area projection.
2. Identify federal parcels that share borders (not just corners)
3. Identify federal patch structure and articulation points (which parcels make up a patch)
4. Select target patches for growth, and exclude small patches not worth growing
5. Identify adjacent non-federal parcels as potential acquire candidates
6. Rank proposals by net gain and visualize the top results on an interactive map.

---
### Reproducibility Notes

---
### ⚡ Troubleshooting/Notes
* 

---

# Libraries

In [15]:
### file paths, OS operations, utilities
import os
import pathlib

### geodata
import geopandas as gpd
from shapely.ops import unary_union

### geospatial plotting
import holoviews as hv
import geoviews as gv
import hvplot.pandas
from cartopy import crs as ccrs

### data handling
import pandas as pd
from collections import defaultdict

### graph analysis
import networkx as nx

### Jupyter display
from IPython.display import display

# Primary Directory

In [16]:
### set up root file path
# Walk up from the current directory to find the repo root (contains .git)
_cwd = pathlib.Path(os.getcwd()).resolve()
repo_root = next(
    (p for p in [_cwd] + list(_cwd.parents) if (p / '.git').exists()),
    _cwd
)
os.chdir(repo_root)

data_dir = os.path.join(repo_root, 'data')
os.makedirs(data_dir, exist_ok=True)

print(f'Repo root: {repo_root}')

Repo root: C:\Users\naho5798\Documents\Earth Data Cert\Final Project\Pawnee-Grasslands-Project


# Secondary Directories

In [17]:
### boundary dirs
boundary_dir       = os.path.join(data_dir, 'boundaries')
boundary_dir_final = os.path.join(boundary_dir, 'boundary-data-final')


### parcel data
parcel_bound      = os.path.join(boundary_dir_final, 'parcel_boundary')
parcel_bound_path = os.path.join(parcel_bound, 'pawnee_parcel.shp')
parcel_bound_gdf  = gpd.read_file(parcel_bound_path)


### output directories
contiguous_dir            = os.path.join(data_dir, 'contiguous')
figures_dir               = os.path.join(repo_root, 'figures')
figures_parcel_matrix_dir = os.path.join(figures_dir, 'parcel_matrix')
os.makedirs(contiguous_dir, exist_ok=True)
os.makedirs(figures_parcel_matrix_dir, exist_ok=True)


### load federal patches from 07_contiguous_area.ipynb output
federal_patches_path = os.path.join(contiguous_dir, 'federal_patches.gpkg')
federal_patches_gdf  = gpd.read_file(federal_patches_path)


print(f'Parcel data loaded : {len(parcel_bound_gdf)} parcels')
print(f'Federal patches    : {len(federal_patches_gdf)} patches')

Parcel data loaded : 3483 parcels
Federal patches    : 166 patches


In [18]:
### load oil/gas parcel join produced by 06_oil_gas.ipynb
oil_gas_processed_dir = os.path.join(data_dir, 'processed', 'oil_gas')
parcel_oil_gas_path   = os.path.join(oil_gas_processed_dir, 'pawnee_parcel_oil_gas.csv')
parcel_oil_gas_df     = pd.read_csv(parcel_oil_gas_path, dtype={'PARCEL': str})

parcel_oil_gas_lookup = dict(zip(parcel_oil_gas_df['PARCEL'], parcel_oil_gas_df['oil_gas_category']))
print(f'Oil/gas lookup loaded: {len(parcel_oil_gas_lookup)} parcels with well activity')

Oil/gas lookup loaded: 204 parcels with well activity


# Classify Ownership

In [19]:
### classify ownership of parcels
def classify_ownership(name):
    if name == 'U S A':
        return 'FEDERAL'
    elif name == 'COLORADO STATE OF':
        return 'STATE'
    else:
        return 'PRIVATE'

### apply to the parcel_bound_gdf
parcel_bound_gdf['ownership'] = parcel_bound_gdf['NAME'].apply(classify_ownership)

# Land Swap Algorithm
We want to identify federal to non-federal land swap proposals that reduce the boundary exposure of federal patches identified in the previous notebook. 

**Algorithm components:**

- **Receive candidates:** Federal patches that could gain private or state land in a swap. In our data, there are lots of these viable candidates. However, some patches are too small to be meaningfully useful for trying to consolidate, and we ignore these by setting a parameter for "small" patches called `SIZE_FLOOR_ACRES = 2000`. 
- **Acquire candidates:** Any non-federal parcel (state or private) sharing a boundary edge with a receive candidate patch.
- **Articulation points:** A federal parcel that connects two or more larger groups of federal parcels. Removing this parcel would break the connection between these federal parcel groups. We do not want to release these from the federal groups as part of a swap.  
- **Release candidates:** Any federal parcel (from any patch) that is not an articulation point, and is within our defined `PROXIMITY_RADIUS_M = 10000` of the acquire parcel. This puts a limit of 5km on how far apart swaps can be. The release candidates also consider a limit called `AREA_TOLERANCE = 0.10` for the difference in area between both parcels in the swap. 

Proposals are one-to-one (one federal parcel traded for one non-federal parcel) and ranked by the net reduction in boundary exposure.

## Analysis Parameters

These parameters can be adjusted as needed to expand or narrow the final swap candidate pool. 

In [20]:
### swap analysis parameters
PROXIMITY_RADIUS_M          = 10000  # 10 km max centroid-to-centroid distance between acquired and released parcel
AREA_TOLERANCE              = 0.10   # hard cutoff, skip if parcels differ > 10% in area
AREA_FLAG                   = 0.05   # soft flag,  warn in summary if parcels differ > 5% in area
MIN_SHARED_M                = 1.0    # minimum shared boundary length (m) to count as adjacent
SIZE_FLOOR_ACRES            = 2000   # receive candidates must meet or exceed this acreage
MIN_SUBPATCH_ACRES          = 3000   # min component size (ac) for a cut point to qualify in sub-patch analysis
SUBPATCH_RATIO_FLOOR        = 0.65   # sub-patch analysis only applies to patches with ratio >= this value
SUBPATCH_PROXIMITY_RADIUS_M = 8000   # larger radius for sub-patch swaps, interior holes pair with distant outer-boundary releases

## Project Data to Metric CRS

We are performing all swap geometry calculations in **EPSG:5070** (NAD83 / Conus Albers), which is an equal-area projection for the continental US. This ensures area comparisons and distance measurements are in meters and acres.

Federal and non-federal parcels are separated into different geo-dataframes, and a geometry lookup dictionary keyed by `PARCEL` ID is built for fast inner-loop access during swap evaluation.

In [21]:
### project all parcel data to EPSG:5070 for swap analysis
parcels_proj = parcel_bound_gdf.to_crs(epsg=5070).copy()

### prepare federal geodataframe
federal_proj  = parcels_proj[parcels_proj['ownership'] == 'FEDERAL'].copy().reset_index(drop=True)

### prepare non-federal geodataframe
nonfed_proj   = parcels_proj[parcels_proj['ownership'] != 'FEDERAL'].copy().reset_index(drop=True)

### dictionary geometry lookup for federal parcels, keyed by PARCEL string
parcel_geom_lookup = federal_proj.set_index('PARCEL')['geometry'].to_dict()

print(f"Federal parcels : {len(federal_proj)}")
print(f"Non-federal parcels (state + private) : {len(nonfed_proj)}")

Federal parcels : 598
Non-federal parcels (state + private) : 2885


## Identify Boundary Edges for Neighboring Parcels

Next, we want to identify all pairs of federal parcels that share a meaningful boundary edge (rook adjacency) with other federal parcels. 

Only `LineString` or `MultiLineString` intersections count as shared edges. Corner-only `Point` touches are ignored.

In [22]:
### build shared edge list for all federal parcels
### uses spatial index to narrow candidates before boundary intersection check
### edges: list of (PARCEL_i, PARCEL_j, shared_length_m)


### set up spatial index (from GeoPandas) for federal patches
sindex_fed = federal_proj.sindex

### list for touching federal edges
fed_edges  = []

### iterate over every parcel
for i, row_i in federal_proj.iterrows():

    ### this finds federal parcels with overlapping grid bounding boxes
    candidates = list(sindex_fed.intersection(row_i.geometry.bounds))
    for j in candidates:

        ### check pairs only once
        if j <= i:
            continue

        ### find the actual geometery between the two parcels
        shared = row_i.geometry.boundary.intersection(federal_proj.geometry[j].boundary)
        
        ### skip if no shared geometry
        if shared.is_empty:
            continue

        ### if two parcels share an edge, measure the length
        if shared.geom_type in ('LineString', 'MultiLineString'):
            length = shared.length

        ### if there are lines and points, we extract just the lines and measure the sum of the lines
        elif shared.geom_type == 'GeometryCollection':
            lines  = [g for g in shared.geoms if g.geom_type in ('LineString', 'MultiLineString')]
            length = sum(g.length for g in lines)
        else:
            continue

        ### filter out very short boundary edges that are basically just touching corners
        if length >= MIN_SHARED_M:
            fed_edges.append((federal_proj.loc[i, 'PARCEL'], federal_proj.loc[j, 'PARCEL'], length))

print(f"Found {len(fed_edges)} federal edge pairs among {len(federal_proj)} parcels.")

Found 598 federal edge pairs among 598 parcels.


## Identify Federal Patch Structure and Articulation Points

Each federal patch is a group of parcels that physically touch each other. We need to identify which parcels belong to which patch.

Once we know this, we can identify articulation points which are parcels that are acting as a bridge holding two parts of a patch together. If you removed one of these parcels, the patch would split into two disconnected pieces. We flag these so the swap algorithm never proposes releasing one.

In [23]:
### make two dictionaries to store parcel IDs from federal_patches_gdf
patch_parcel_sets = {} # given a patch ID, what parcels are in it?
parcel_to_patch = {}   # given a parcel ID, which patch does it belong to?

### iterate over the patches
for _, row in federal_patches_gdf.iterrows():
    
    ### get patch ID
    pid         = row['contig_parcel_id']

    ### get parcel IDs
    parcels_val = row['parcels']

    ### skip single parcel patches 
    if not isinstance(parcels_val, str):
        continue   # skip patches where parcels column is NaN

    ### split comma-separated string into a set of individual parcel IDs
    parcels = {p.strip() for p in parcels_val.split(',')}

    ### store under patch ID
    patch_parcel_sets[pid] = parcels

    ### save corresponding patch ID for each parcel
    for p in parcels:
        parcel_to_patch[p] = pid

### per-patch edge lists and interior sums

### dictionary where each key is a patch ID and each value is a list
patch_edges = defaultdict(list)

### loop over each fed shared edge neighbor
for p1, p2, length in fed_edges:

    ### find which patch parcel "p1" belongs to 
    patch = parcel_to_patch.get(p1)

    ### check if p1 belongs to a patch and that p2 belongs to the same patch
    if patch and parcel_to_patch.get(p2) == patch:

        ### add the edge to the patch list
        patch_edges[patch].append((p1, p2, length))

### for each patch, add up all the shared boundary lengths across its interior edges
patch_interior_sum = {
    pid: sum(l for _, _, l in edges)
    for pid, edges in patch_edges.items()
}


### Find articulation points

### dict for graph for each patch (graph of the connections between each parcel in a patch)
patch_graphs     = {}

### dict for articulation points
patch_art_points = {}

### loop over every patch and its parcel IDs
for pid, parcel_ids in patch_parcel_sets.items():
    
    ### create a new NetworkX graph from the NetworkX library
    G = nx.Graph()

    ### add each parcel as a node in the graph
    G.add_nodes_from(parcel_ids)

    ### add an edge between every pair of parcels that share a boundary 
    for p1, p2, _ in patch_edges[pid]:
        G.add_edge(p1, p2)

    ### save the graph, and use nx.articulation_points
    patch_graphs[pid] = G
    patch_art_points[pid] = set(nx.articulation_points(G))

### count the number of articulation points
n_art = sum(len(v) for v in patch_art_points.values())
print(f"Parsed {len(patch_parcel_sets)} patches. {n_art} articulation points identified.")

Parsed 148 patches. 133 articulation points identified.


## Select Target Patches for Growth

Not all federal patches are worth trying to grow. We want to ignore small patches that are not worth growing. Any multi-parcel federal patch meeting or exceeding `SIZE_FLOOR_ACRES` is included as a receive candidate.

In [24]:
### select receive candidate patches: all multi-parcel patches >= SIZE_FLOOR_ACRES

### filter to only patches with more than one parcel
patches_multi = federal_patches_gdf[federal_patches_gdf['n_parcels'] > 1].copy()

### keep only patches that meet or exceed the minimum acreage threshold
receive_candidates = patches_multi[
    patches_multi['area_acres'] >= SIZE_FLOOR_ACRES
].reset_index(drop=True)

### print the list of receive candidates
print(f"{len(receive_candidates)} receive candidates (>= {SIZE_FLOOR_ACRES} ac):")
print(
    receive_candidates[['contig_parcel_id', 'area_acres', 'n_parcels', 'interior_fraction']]
    .to_string(index=False)
)

19 receive candidates (>= 2000 ac):
contig_parcel_id   area_acres  n_parcels  interior_fraction
       PATCH_001 58397.167533      118.0             0.4199
       PATCH_002 16043.807741       34.0             0.4009
       PATCH_003 15766.553980       54.0             0.3927
       PATCH_004 14462.476212       28.0             0.3462
       PATCH_005  8032.243481       15.0             0.3374
       PATCH_006  8018.897989       15.0             0.3705
       PATCH_007  7082.598887       12.0             0.3951
       PATCH_008  5924.534925       14.0             0.3920
       PATCH_009  4879.949564       13.0             0.3917
       PATCH_010  4309.266388       11.0             0.2987
       PATCH_011  4026.737934       10.0             0.2573
       PATCH_012  3325.158556        8.0             0.3329
       PATCH_013  3038.094044       12.0             0.4626
       PATCH_014  2735.784096        6.0             0.3555
       PATCH_015  2711.659739        5.0             0.2312
    

## Identify Adjacent Non-Federal Parcels

For each receive-candidate patch, we want to find all non-federal (state or private) parcels that share a rook edge with any parcel in the patch. These are the candidate parcels to acquire. Adding one of these to a federal patch increases its area and, if well-positioned, its interior edge ratio.

In [25]:
### find non-federal parcels sharing a boundary edge with any parcel in each receive candidate patch
### adj_edges: (patch_id, nf_idx) -> list of (fed_parcel_id, shared_length_m)

### build spatial index for non-federal parcels
sindex_nonfed = nonfed_proj.sindex

### dictionary to store adjacent (patch, non-federal parcel) pairs
adj_edges = defaultdict(list)

### loop over each receive candidate patch
for patch_id in receive_candidates['contig_parcel_id']:

    ### loop over each federal parcel in the patch
    for fed_parcel_id in patch_parcel_sets[patch_id]:

        ### get the geometry of the federal parcel
        fed_geom = parcel_geom_lookup[fed_parcel_id]

        ### use spatial index to find nearby non-federal parcels
        candidates = list(sindex_nonfed.intersection(fed_geom.bounds))

        ### loop over each nearby non-federal parcel
        for nf_idx in candidates:
            nf_geom = nonfed_proj.geometry[nf_idx]

            ### compute the shared boundary between the federal and non-federal parcel
            shared = fed_geom.boundary.intersection(nf_geom.boundary)

            ### skip if no shared geometry
            if shared.is_empty:
                continue

            ### if they share an edge, measure its length
            if shared.geom_type in ('LineString', 'MultiLineString'):
                length = shared.length

            ### if the result is a mix of lines and points, extract just the lines
            elif shared.geom_type == 'GeometryCollection':
                lines  = [g for g in shared.geoms if g.geom_type in ('LineString', 'MultiLineString')]
                length = sum(g.length for g in lines)
            else:
                continue

            ### only count as adjacent if the shared edge is long enough
            if length >= MIN_SHARED_M:
                adj_edges[(patch_id, nf_idx)].append((fed_parcel_id, length))

### count unique non-federal parcels found
n_unique_nf = len({nf_idx for _, nf_idx in adj_edges})
print(f"Found {len(adj_edges)} (patch, non-federal parcel) adjacent pairs "
      f"({n_unique_nf} unique non-federal parcels).")

Found 610 (patch, non-federal parcel) adjacent pairs (594 unique non-federal parcels).


## Evaluate Swap Proposals

For each receive patch and adjacent non-federal parcel pair, we want to search for a releasable federal parcel to trade in return. A federal parcel qualifies as a release candidate if:

1. It is not an articulation point in its own patch
2. It is within `PROXIMITY_RADIUS_M` of the acquired parcel
3. Its area is within `AREA_TOLERANCE` of the acquired parcel's area
4. It touches at least one non-federal parcel
5. It has fewer than 2 federal adjacent neighbors

The new boundary exposure for the receive patch is computed after the hypothetical swap. Only proposals where the exposure decreases are retained. Release patch impacts are cached to avoid redundant dissolve operations on the same (patch, parcel) combination.

In [26]:
### evaluate swap proposals
### for each receive patch + adjacent non-federal parcel, find releasable federal parcels
### release parcel may come from any patch (cross-patch swap) as long as:
###   1. not an articulation point in its own patch
###   2. centroid within PROXIMITY_RADIUS_M of the non-federal parcel's centroid
###   3. area within AREA_TOLERANCE of the non-federal parcel's area
###   4. touches at least one non-federal parcel (layer 1)
###   5. does not have 2+ federal rook-neighbors (layer 2)

### store the dissolved geometry of each patch (used to compute new perimeter after swap)
patch_dissolved_geom = {
    row['contig_parcel_id']: row['geometry']
    for _, row in federal_patches_gdf.iterrows()
}

### store the current interior fraction of each patch
patch_old_ratio = {
    row['contig_parcel_id']: row['interior_fraction']
    for _, row in federal_patches_gdf.iterrows()
}

### build a fast lookup dict of federal parcel geometry, area, centroid, and patch membership
fed_info = {
    row['PARCEL']: {
        'geom':     row['geometry'],
        'area':     row['geometry'].area,
        'centroid': row['geometry'].centroid,
        'patch_id': parcel_to_patch.get(row['PARCEL']),
    }
    for _, row in federal_proj.iterrows()
}

### build a spatial index on federal parcel centroids for proximity pre-filtering
fed_centroid_gdf             = federal_proj.copy()
fed_centroid_gdf['geometry'] = federal_proj.geometry.centroid
fed_centroid_sindex          = fed_centroid_gdf.sindex

### layer 2: count federal rook-neighbors per parcel
### parcels with 2+ federal neighbors are too interior to safely release
_fed_neighbor_count = {}
for _p1, _p2, _ in fed_edges:
    _fed_neighbor_count[_p1] = _fed_neighbor_count.get(_p1, 0) + 1
    _fed_neighbor_count[_p2] = _fed_neighbor_count.get(_p2, 0) + 1

### layer 1: precompute which federal parcels touch at least one non-federal parcel
### purely interior parcels (surrounded only by federal land) cannot be released
fed_on_boundary = set()
for _fp, _geom in parcel_geom_lookup.items():
    for _nf_idx in sindex_nonfed.intersection(_geom.bounds):
        _sh = _geom.boundary.intersection(nonfed_proj.geometry[_nf_idx].boundary)
        if _sh.is_empty:
            continue
        if _sh.geom_type in ('LineString', 'MultiLineString'):
            fed_on_boundary.add(_fp)
            break
        elif _sh.geom_type == 'GeometryCollection':
            if any(g.geom_type in ('LineString', 'MultiLineString') for g in _sh.geoms):
                fed_on_boundary.add(_fp)
                break

### bridge lookup: for each NF parcel index, which distinct receive patches is it adjacent to?
nf_patch_adjacency = defaultdict(set)
for (_patch_id, _nf_idx) in adj_edges.keys():
    nf_patch_adjacency[_nf_idx].add(_patch_id)

### all-patches area lookup used for bridge_gain_acres — includes sub-floor patches so that
### bridging into a small patch below SIZE_FLOOR_ACRES still gets a real area estimate
patch_area_lookup_all = federal_patches_gdf.set_index('contig_parcel_id')['area_acres'].to_dict()

### cache to avoid recomputing the same release patch ratio multiple times
release_cache = {}

### list to collect all valid swap proposals
proposals = []
print("Evaluating swap proposals...")

### loop over every (receive patch, adjacent non-federal parcel) pair
for (patch_id, nf_idx), edges_to_patch in adj_edges.items():

    ### get geometry and properties of the non-federal parcel being considered for acquisition
    nf_geom         = nonfed_proj.geometry[nf_idx]
    nf_area         = nf_geom.area
    nf_centroid     = nf_geom.centroid
    nf_acreage      = nf_area * 0.000247105

    ### get current ratio and interior edge sum for the receive patch
    old_ratio       = patch_old_ratio[patch_id]
    old_interior    = patch_interior_sum.get(patch_id, 0.0)
    old_exposed_m   = patch_dissolved_geom[patch_id].length

    ### total shared boundary length between the non-federal parcel and the receive patch
    new_interior_perim = sum(l for _, l in edges_to_patch)

    ### acquisition exposure: fraction of NF parcel boundary already touching federal land
    nf_perim     = nf_geom.length
    acq_fraction = new_interior_perim / nf_perim if nf_perim > 0 else 0.0

    ### bridge detection: does this NF parcel touch multiple distinct receive patches?
    n_patches_bridged     = len(nf_patch_adjacency[nf_idx])
    bridges               = n_patches_bridged > 1

    ### bridge acreage credit: min of all connected patch areas (including receive patch),
    ### capping so a bridge into an already-large patch doesn't over-inflate the score
    bridge_gain_acres = 0.0
    bridge_perimeter_m = 0.0
    if bridges:
        connected_areas    = [patch_area_lookup_all.get(pid, 0.0) for pid in nf_patch_adjacency[nf_idx]]
        bridge_gain_acres  = min(connected_areas)
        bridge_perimeter_m = sum(
            sum(l for _, l in adj_edges[(pid, nf_idx)])
            for pid in nf_patch_adjacency[nf_idx]
        )

    ### precompute the new geometry and perimeter if nf parcel is added to the receive patch
    ### used for cross-patch swaps where the release parcel comes from a different patch
    recv_with_nf_geom      = patch_dissolved_geom[patch_id].union(nf_geom)
    recv_with_nf_perimeter = recv_with_nf_geom.length
    recv_with_nf_interior  = old_interior + new_interior_perim

    ### temporarily add the non-federal parcel to the receive patch graph
    ### so we can check if any federal parcel becomes an articulation point after the swap
    G_aug = patch_graphs[patch_id].copy()
    G_aug.add_node('_nf_')
    for fp, _ in edges_to_patch:
        G_aug.add_edge('_nf_', fp)
    aug_art_points = set(nx.articulation_points(G_aug))

    ### use spatial index to find federal parcels within PROXIMITY_RADIUS_M of the nf parcel
    search_buffer = nf_centroid.buffer(PROXIMITY_RADIUS_M)
    nearby_rows   = list(fed_centroid_sindex.intersection(search_buffer.bounds))

    ### loop over each nearby federal parcel as a potential release candidate
    for row_idx in nearby_rows:
        f_parcel_id = federal_proj.loc[row_idx, 'PARCEL']
        f           = fed_info[f_parcel_id]
        f_patch_id  = f['patch_id']

        ### skip if the parcel doesn't belong to any patch
        if f_patch_id is None:
            continue

        ### skip if the parcel is outside the exact proximity radius
        if nf_centroid.distance(f['centroid']) > PROXIMITY_RADIUS_M:
            continue

        ### skip if the area difference between the two parcels exceeds AREA_TOLERANCE
        area_diff = abs(nf_area - f['area']) / max(nf_area, f['area'])
        if area_diff > AREA_TOLERANCE:
            continue

        ### layer 1: release must touch non-federal land — interior parcels would create an enclave
        if f_parcel_id not in fed_on_boundary:
            continue

        ### layer 2: release touches 2+ federal parcels — too interior, would create a hole
        if _fed_neighbor_count.get(f_parcel_id, 0) >= 2:
            continue

        ### check if the release parcel is in the same patch as the receive patch
        same_patch = (f_patch_id == patch_id)

        if same_patch:
            ### skip if releasing this parcel would make the patch disconnected
            if f_parcel_id in aug_art_points:
                continue

            ### compute the new interior edge sum after removing the release parcel and adding the nf parcel
            lost_interior_perim = sum(l for p1, p2, l in patch_edges[patch_id]
                                      if p1 == f_parcel_id or p2 == f_parcel_id)
            new_interior        = old_interior - lost_interior_perim + new_interior_perim

            ### compute the new perimeter by dissolving the remaining parcels plus the nf parcel
            remaining_geoms = (
                [parcel_geom_lookup[p] for p in patch_parcel_sets[patch_id] if p != f_parcel_id]
                + [nf_geom]
            )
            new_perimeter     = unary_union(remaining_geoms).length
            new_exposed_m     = new_perimeter
            new_ratio         = new_interior / (new_interior + new_perimeter) if (new_interior + new_perimeter) > 0 else 0.0
            release_old_ratio = old_ratio
            release_new_ratio = new_ratio   # same patch — receive and release are one operation

        else:
            ### cross-patch swap: skip if releasing this parcel would disconnect its own patch
            if f_parcel_id in patch_art_points.get(f_patch_id, set()):
                continue

            ### new ratio for the receive patch after gaining the nf parcel
            new_ratio = recv_with_nf_interior / (recv_with_nf_interior + recv_with_nf_perimeter) if (recv_with_nf_interior + recv_with_nf_perimeter) > 0 else 0.0
            new_exposed_m = recv_with_nf_perimeter

            ### compute lost_interior_perim for this release parcel (needed for rel_fraction)
            lost_interior_perim = sum(l for p1, p2, l in patch_edges[f_patch_id]
                                      if p1 == f_parcel_id or p2 == f_parcel_id)

            ### compute and cache the new ratio for the release patch after losing this parcel
            cache_key = (f_patch_id, f_parcel_id)
            if cache_key not in release_cache:
                rel_old_interior = patch_interior_sum.get(f_patch_id, 0.0)
                rel_remaining    = [parcel_geom_lookup[p]
                                    for p in patch_parcel_sets[f_patch_id] if p != f_parcel_id]
                if rel_remaining:
                    rel_perimeter            = unary_union(rel_remaining).length
                    rel_new_interior         = rel_old_interior - lost_interior_perim
                    release_cache[cache_key] = rel_new_interior / (rel_new_interior + rel_perimeter) if (rel_new_interior + rel_perimeter) > 0 else 0.0
                else:
                    release_cache[cache_key] = 0.0   # single-parcel patch fully traded away

            release_old_ratio = patch_old_ratio[f_patch_id]
            release_new_ratio = release_cache[cache_key]

        ### release exposure: fraction of released federal parcel boundary that is interior to its patch
        fed_perim    = parcel_geom_lookup[f_parcel_id].length
        rel_fraction = lost_interior_perim / fed_perim if fed_perim > 0 else 0.0

        ### parcel exposure delta: positive = good trade (acquiring enclosed parcel, releasing exposed parcel)
        parcel_exposure_delta = acq_fraction - rel_fraction

        ### compute net gain in interior fraction for the receive patch
        net_gain = new_ratio - old_ratio

        ### skip proposals that don't improve the receive patch ratio
        if net_gain <= 0:
            continue

        ### net acreage trade (acquired minus released) plus bridge credit capped at smaller patch
        fed_acreage           = f['area'] * 0.000247105
        area_gain_acres       = nf_acreage - fed_acreage
        contiguity_gain_acres = area_gain_acres + bridge_gain_acres

        ### save the valid proposal
        proposals.append(dict(
            receive_patch_id  = patch_id,
            old_ratio         = round(old_ratio, 4),
            new_ratio         = round(new_ratio, 4),
            net_gain          = round(net_gain, 4),
            old_exposed_m     = round(old_exposed_m, 1),
            new_exposed_m     = round(new_exposed_m, 1),
            nf_parcel_id      = nonfed_proj.loc[nf_idx, 'PARCEL'],
            nf_ownership      = nonfed_proj.loc[nf_idx, 'ownership'],
            nf_acres          = round(nf_acreage, 1),
            release_parcel_id = f_parcel_id,
            release_patch_id  = f_patch_id,
            fed_acres         = round(fed_acreage, 1),
            area_diff_pct     = round(area_diff * 100, 1),
            area_flag         = area_diff > AREA_FLAG,
            distance_km       = round(nf_centroid.distance(f['centroid']) / 1000, 2),
            same_patch            = same_patch,
            release_old_ratio     = round(release_old_ratio, 4),
            release_new_ratio     = round(release_new_ratio, 4),
            bridges               = bridges,
            n_patches_bridged     = n_patches_bridged,
            contiguity_gain_acres = round(contiguity_gain_acres, 1),
            bridge_perimeter_m    = round(bridge_perimeter_m, 1),
            acq_fraction          = round(acq_fraction, 4),
            rel_fraction          = round(rel_fraction, 4),
            parcel_exposure_delta = round(parcel_exposure_delta, 4),
        ))

print(f"Found {len(proposals)} valid swap proposals.")

Evaluating swap proposals...
Found 703 valid swap proposals.


## Results Summary

Proposals are ranked by three metrics in priority order:

1. **Bridge swaps first** — proposals where the acquired parcel connects two or more separate federal patches rank above all others
2. **Contiguity gain (acres)** — total contiguous federal acres added by the swap (Non-federal (NF) parcel acreage + merged patch acreage for bridge swaps)
3. **Patch exposure reduction** — decrease in boundary exposure (`1 - interior_fraction`) for the receive patch after the swap. A positive `net_gain` in `interior_fraction` means exposure went down. Higher `net_gain` values = more exposure reduced.
4. **Parcel exposure delta** (`acq_fraction − rel_fraction`) — tiebreaker measuring how much more enclosed the acquired parcel is relative to the released parcel. 

Each proposal reports:

- Which patch gains land and how its boundary exposure (`1 - interior_fraction`) is reduced, plus contiguity acres gained
- Whether the swap bridges two separate federal patches
- The non-federal parcel being acquired: ownership type, acreage, and `acq_fraction` (fraction of its boundary already touching federal land)
- The federal parcel being released: acreage, source patch, and `rel_fraction` (fraction of its boundary that is interior to its patch)
- Acreage difference and centroid-to-centroid distance between the two parcels
- Release patch ratio before and after (cross-patch swaps only)
- Landowner contiguity gain flag (`landowner_contiguity_gain`): whether the released federal parcel is next to other land held by the same non-federal party. By specific `NAME` for private owners, or any state parcel for state ownership.

In [27]:
### print ranked summary of swap proposals

### if no proposals were found, print a message and stop
if not proposals:
    print("No valid swap proposals found.")
else:
    ### convert proposals list to a dataframe
    proposals_raw_df = pd.DataFrame(proposals)

    ### blended bridge-priority score: for bridge swaps only, combine contiguity gain
    ### and bridge connection width into one score. Each is scaled 0-1 by its own max
    ### across bridge proposals, then averaged so the combined score stays in 0-1 —
    ### a bridge into a slightly larger patch doesn't automatically outrank a much
    ### wider, sturdier connection, or vice versa
    is_bridge       = proposals_raw_df['bridges']
    max_gain_acres  = proposals_raw_df.loc[is_bridge, 'contiguity_gain_acres'].max() if is_bridge.any() else 0.0
    max_perimeter_m = proposals_raw_df.loc[is_bridge, 'bridge_perimeter_m'].max()    if is_bridge.any() else 0.0

    bridge_connection_score = pd.Series(0.0, index=proposals_raw_df.index)
    if is_bridge.any():
        gain_norm  = proposals_raw_df.loc[is_bridge, 'contiguity_gain_acres'] / max_gain_acres  if max_gain_acres  > 0 else 0.0
        perim_norm = proposals_raw_df.loc[is_bridge, 'bridge_perimeter_m']    / max_perimeter_m if max_perimeter_m > 0 else 0.0
        bridge_connection_score.loc[is_bridge] = (gain_norm + perim_norm) / 2

    proposals_raw_df['bridge_connection_score'] = bridge_connection_score.round(4)

    ### rank keys differ by group (bridges vs non-bridges are never compared against each other):
    ###   bridges:     bridge_connection_score → contiguity_gain_acres → net_gain → parcel_exposure_delta
    ###   non-bridges: contiguity_gain_acres → net_gain → parcel_exposure_delta
    proposals_raw_df['rank_key']   = proposals_raw_df['bridge_connection_score'].where(is_bridge, proposals_raw_df['contiguity_gain_acres'])
    proposals_raw_df['second_key'] = proposals_raw_df['contiguity_gain_acres'].where(is_bridge, proposals_raw_df['net_gain'])

    proposals_df = (
        proposals_raw_df
        .sort_values(
            ['bridges', 'rank_key', 'second_key', 'net_gain', 'parcel_exposure_delta'],
            ascending=[False, False, False, False, False]
        )
        .drop(columns=['rank_key', 'second_key'])
        .reset_index(drop=True)
    )

    ### start ranks at 1 instead of 0
    proposals_df.index += 1

    ### post-processing: flag proposals where the released federal parcel
    ### connects to existing landholdings of the same non-federal party

    ### derive landowner NAME from nonfed_proj using the NF parcel ID
    nf_name_lookup = nonfed_proj.set_index('PARCEL')['NAME'].to_dict()
    proposals_df['nf_name'] = proposals_df['nf_parcel_id'].map(nf_name_lookup)

    def check_landowner_contiguity(row):
        """
        Returns True if the released federal parcel shares a boundary edge
        with another parcel owned by the same NAME 
        (for private land uses the landowner NAME) as the acquired NF parcel.
        The NF parcel being given up is excluded (it is no longer theirs after the swap).
        Uses sindex_nonfed as a lookup for non-federal parcel locations.
        """
        nf_name      = row['nf_name']
        release_geom = parcel_geom_lookup.get(row['release_parcel_id'])
        if not nf_name or release_geom is None:
            return False
        for idx in sindex_nonfed.intersection(release_geom.bounds):
            if nonfed_proj.loc[idx, 'NAME']   != nf_name:             continue
            if nonfed_proj.loc[idx, 'PARCEL'] == row['nf_parcel_id']: continue
            shared = release_geom.boundary.intersection(nonfed_proj.geometry[idx].boundary)
            if not shared.is_empty and shared.geom_type in ('LineString', 'MultiLineString'):
                return True
            if shared.geom_type == 'GeometryCollection':
                if any(g.geom_type in ('LineString', 'MultiLineString') for g in shared.geoms):
                    return True
        return False

    proposals_df['landowner_contiguity_gain'] = proposals_df.apply(
        check_landowner_contiguity, axis=1
    )
    n_lcg = proposals_df['landowner_contiguity_gain'].sum()
    print(f"Proposals with landowner contiguity gain: {n_lcg} of {len(proposals_df)}")

    ### oil/gas flags: which category of well activity is present on each parcel?
    proposals_df['acquire_oil_gas_flag'] = proposals_df['nf_parcel_id'].map(
        lambda pid: parcel_oil_gas_lookup.get(str(pid))
    )
    proposals_df['release_oil_gas_flag'] = proposals_df['release_parcel_id'].map(
        lambda pid: parcel_oil_gas_lookup.get(str(pid))
    )

    n_acq_og = proposals_df['acquire_oil_gas_flag'].notna().sum()
    n_rel_og = proposals_df['release_oil_gas_flag'].notna().sum()
    print(f'Proposals with O&G on acquire parcel : {n_acq_og}')
    print(f'Proposals with O&G on release parcel : {n_rel_og}')

    ### export the full ranked table for script 09 (avoids Jupyter stdout truncation)
    proposals_export = proposals_df.reset_index().rename(columns={'index': 'rank'})
    for col in ('nf_parcel_id', 'release_parcel_id'):
        proposals_export[col] = proposals_export[col].astype(str)
    proposals_csv = pathlib.Path('data/processed/parcel_swaps/pawnee_land_swap_proposals.csv')
    proposals_csv.parent.mkdir(parents=True, exist_ok=True)
    proposals_export.to_csv(proposals_csv, index=False)
    print(f'Saved {len(proposals_export)} proposals to {proposals_csv.resolve()}')

    print("=" * 70)
    print("  LAND SWAP PROPOSAL SUMMARY")
    print("=" * 70)

    ### print only the top proposals for readability; full set is in the CSV
    PRINT_TOP_N = 30

    for rank, row in proposals_df.head(PRINT_TOP_N).iterrows():

        ### build a flag string if the area difference exceeds the soft warning threshold
        flags = []
        if row['area_flag']:
            flags.append(
                f"AREA DIFFERENCE {row['area_diff_pct']}%  "
                f"(acquire {row['nf_acres']} ac / release {row['fed_acres']} ac)"
            )
        flag_str = ("\n    [!]  " + "\n    [!]  ".join(flags)) if flags else ""

        ### label the swap as same-patch or cross-patch
        swap_label = "same-patch" if row['same_patch'] else "cross-patch"

        ### print proposal header with bridge flag if applicable
        bridge_tag = "  [BRIDGE]" if row['bridges'] else ""
        print(f"\n  Proposal #{rank}  |  {row['receive_patch_id']}  [{swap_label}]{bridge_tag}")

        ### print the ranking metrics
        print(f"    Contiguity gain     : +{row['contiguity_gain_acres']:.1f} ac")
        if row['bridges']:
            print(f"    Bridge width        : {row['bridge_perimeter_m']:.1f} m"
                  f"  (connection score {row['bridge_connection_score']:.4f})")
        print(f"    Receive patch exposure: {(1-row['old_ratio'])*100:.1f}%  ->  {(1-row['new_ratio'])*100:.1f}%  (\u2212{row['net_gain']*100:.2f} pp)")
        print(f"    Parcel exposure delta: {row['parcel_exposure_delta']:+.4f}"
              f"  (acq {row['acq_fraction']:.4f} / rel {row['rel_fraction']:.4f})")

        ### print the acquire and release parcels
        print(f"    Acquire  : {row['nf_ownership']} parcel  {row['nf_parcel_id']}  ({row['nf_acres']} ac)")
        print(f"    Release  : FEDERAL parcel  {row['release_parcel_id']}  ({row['fed_acres']} ac)  "
              f"from {row['release_patch_id']}")

        ### print area difference and distance between the two parcels
        acreage_diff = abs(row['nf_acres'] - row['fed_acres'])
        print(f"    Area difference     : {row['area_diff_pct']}%  ({acreage_diff:.1f} ac)")
        print(f"    Distance            : {row['distance_km']} km")

        ### for cross-patch swaps, also show the impact on the release patch ratio
        if not row['same_patch']:
            print(f"    Release patch exposure: {(1-row['release_old_ratio'])*100:.1f}%  ->  {(1-row['release_new_ratio'])*100:.1f}%")

        ### print landowner contiguity flag if applicable
        if row['landowner_contiguity_gain']:
            print(f"    [+] Landowner contiguity: {row['nf_name']} receives land adjacent to existing holdings")

        ### print oil/gas activity flags if applicable
        if pd.notna(row['acquire_oil_gas_flag']) and row['acquire_oil_gas_flag']:
            print(f"    [OG-ACQ] O&G activity on acquire parcel: {row['acquire_oil_gas_flag']}")
        if pd.notna(row['release_oil_gas_flag']) and row['release_oil_gas_flag']:
            print(f"    [OG-REL] O&G activity on release parcel: {row['release_oil_gas_flag']}")

        ### print any area difference warnings
        if flag_str:
            print(flag_str)

    ### print totals
    print("\n" + "=" * 70)
    print(f"\nTotal proposals found : {len(proposals_df)}")
    print(f"Shown above           : top {min(PRINT_TOP_N, len(proposals_df))}")
    print(f"Full export           : {proposals_csv}")

Proposals with landowner contiguity gain: 106 of 703
Proposals with O&G on acquire parcel : 37
Proposals with O&G on release parcel : 8
Saved 703 proposals to C:\Users\naho5798\Documents\Earth Data Cert\Final Project\Pawnee-Grasslands-Project\data\processed\parcel_swaps\pawnee_land_swap_proposals.csv
  LAND SWAP PROPOSAL SUMMARY

  Proposal #1  |  PATCH_001  [same-patch]  [BRIDGE]
    Contiguity gain     : +14492.4 ac
    Bridge width        : 4064.9 m  (connection score 0.8153)
    Receive patch exposure: 58.0%  ->  57.6%  (−0.45 pp)
    Parcel exposure delta: +0.1263  (acq 0.3760 / rel 0.2497)
    Acquire  : PRIVATE parcel  029729000009  (649.2 ac)
    Release  : FEDERAL parcel  020734000004  (619.3 ac)  from PATCH_001
    Area difference     : 4.6%  (29.9 ac)
    Distance            : 8.76 km

  Proposal #2  |  PATCH_001  [same-patch]  [BRIDGE]
    Contiguity gain     : +14491.3 ac
    Bridge width        : 4064.9 m  (connection score 0.8153)
    Receive patch exposure: 58.0%  ->  5

## Swap Map


- **Yellow patches:** federal contiguous patches (multi-parcel only)
- **Green parcels:** non-federal land proposed for acquisition, labeled A#
- **Red parcels:** federal land proposed for release, labeled R#
- Parcels shared across multiple proposals carry a combined label (e.g., A1/A3)


In [28]:
if 'proposals_df' not in dir() or proposals_df.empty:
    print("No proposals to map.")
else:
    import geoviews.tile_sources as gvts
    from bokeh.models import GlyphRenderer, Legend, LegendItem, ColumnDataSource
    from bokeh.models.glyphs import MultiPolygons as _MP
    from bokeh.resources import CDN
    from bokeh.embed import file_html, components
    from IPython.display import HTML

    ### number of top proposals to display on the map
    TOP_N         = len(proposals_df)

    ### color scheme for each parcel category
    ACQUIRE_COLOR = '#2ca25f'
    RELEASE_COLOR = '#d7301f'
    PATCH_COLOR   = '#fff29b'

    ### slice the top N proposals from the ranked dataframe
    plot_proposals = proposals_df.head(TOP_N)

    ### reproject all polygon layers to EPSG:3857 (native tile CRS)
    ### gv.Polygons(crs=ccrs.GOOGLE_MERCATOR) declares input = display CRS so project_path
    ### performs an identity transform instead of the buggy ellipsoidal→spherical conversion
    ### that shifts 4326 polygons ~28 km north of the tile background at ~41°N
    parcels_3857 = parcel_bound_gdf.to_crs(epsg=3857)
    patches_3857 = (
        federal_patches_gdf[federal_patches_gdf['n_parcels'] > 1]
        .to_crs(epsg=3857)
        [['contig_parcel_id', 'area_acres', 'n_parcels', 'interior_fraction', 'geometry']]
        .reset_index(drop=True)
    )

    ### fast geometry lookup for non-federal parcels keyed by PARCEL string
    nonfed_geom_idx = nonfed_proj.set_index('PARCEL')['geometry']

    ### build acquire and release GDFs from the top proposals (still in EPSG:5070)
    acq_5070 = gpd.GeoDataFrame(
        geometry=[nonfed_geom_idx[r['nf_parcel_id']] for _, r in plot_proposals.iterrows()],
        crs='EPSG:5070'
    )
    rel_5070 = gpd.GeoDataFrame(
        geometry=[parcel_geom_lookup[r['release_parcel_id']] for _, r in plot_proposals.iterrows()],
        crs='EPSG:5070'
    )

    ### reproject acquire and release geometries to 3857 for map display
    acq_3857_all = acq_5070.to_crs(3857)
    rel_3857_all = rel_5070.to_crs(3857)

    ### deduplicate acquire parcels — the same parcel may appear in multiple proposals
    ### combine labels (e.g. A1/A3) so a single polygon carries all relevant rank numbers
    seen_acq = {}
    for i, (rank, row) in enumerate(plot_proposals.iterrows()):
        pid  = row['nf_parcel_id']
        cent = gpd.GeoSeries([acq_5070.geometry.iloc[i].centroid], crs=5070).to_crs(3857).iloc[0]
        if pid not in seen_acq:
            seen_acq[pid] = dict(
                geometry=acq_3857_all.geometry.iloc[i], label=f'A{rank}',
                label_x=cent.x, label_y=cent.y,
                ownership=row['nf_ownership'], name=row['nf_name'],
                landowner_contiguity_gain=row['landowner_contiguity_gain'],
                acres=row['nf_acres'], proposals=str(rank),
            )
        else:
            seen_acq[pid]['label']     += f'/A{rank}'
            seen_acq[pid]['proposals'] += f', {rank}'

    ### deduplicate release parcels in the same way
    seen_rel = {}
    for i, (rank, row) in enumerate(plot_proposals.iterrows()):
        pid  = row['release_parcel_id']
        cent = gpd.GeoSeries([rel_5070.geometry.iloc[i].centroid], crs=5070).to_crs(3857).iloc[0]
        if pid not in seen_rel:
            seen_rel[pid] = dict(
                geometry=rel_3857_all.geometry.iloc[i], label=f'R{rank}',
                label_x=cent.x, label_y=cent.y,
                release_patch=row['release_patch_id'], acres=row['fed_acres'], proposals=str(rank),
            )
        else:
            seen_rel[pid]['label']     += f'/R{rank}'
            seen_rel[pid]['proposals'] += f', {rank}'

    ### convert deduplicated dicts to GDFs for plotting
    acq_unique = gpd.GeoDataFrame(list(seen_acq.values()), geometry='geometry', crs='EPSG:3857')
    rel_unique = gpd.GeoDataFrame(list(seen_rel.values()), geometry='geometry', crs='EPSG:3857')

    ### base tile layer and transparent parcel outline layer
    tiles = gvts.CartoLight.opts(width=950, height=680)

    base = gv.Polygons(parcels_3857, vdims=['PARCEL'], crs=ccrs.GOOGLE_MERCATOR).opts(
        line_color='gray', line_width=0.3, fill_alpha=0,
    )

    ### federal patch fill layer (amber, semi-transparent)
    patch_layer = gv.Polygons(
        patches_3857,
        vdims=['contig_parcel_id', 'area_acres', 'n_parcels', 'interior_fraction'],
        crs=ccrs.GOOGLE_MERCATOR,
    ).opts(color=PATCH_COLOR, line_color='black', line_width=0.6, fill_alpha=0.65, tools=['hover'])

    ### acquire layer (green)
    acq_layer = gv.Polygons(
        acq_unique[['label', 'ownership', 'name', 'landowner_contiguity_gain', 'acres', 'proposals', 'geometry']],
        vdims=['label', 'ownership', 'name', 'landowner_contiguity_gain', 'acres', 'proposals'],
        crs=ccrs.GOOGLE_MERCATOR,
    ).opts(color=ACQUIRE_COLOR, line_color='black', line_width=1.5, fill_alpha=0.9, tools=['hover'])

    ### release layer (red)
    rel_layer = gv.Polygons(
        rel_unique[['label', 'release_patch', 'acres', 'proposals', 'geometry']],
        vdims=['label', 'release_patch', 'acres', 'proposals'],
        crs=ccrs.GOOGLE_MERCATOR,
    ).opts(color=RELEASE_COLOR, line_color='black', line_width=1.5, fill_alpha=0.9, tools=['hover'])

    ### shared label style for all parcel rank labels
    _label_opts = dict(
        text_font_size='8pt', text_color='white',
        text_font_style='bold', text_align='center', text_baseline='middle',
    )

    ### rank labels centered on each unique acquire parcel
    acq_labels = hv.Overlay([
        gv.Text(row['label_x'], row['label_y'], row['label'], crs=ccrs.GOOGLE_MERCATOR).opts(**_label_opts)
        for _, row in acq_unique.iterrows()
    ])

    ### rank labels centered on each unique release parcel
    rel_labels = hv.Overlay([
        gv.Text(row['label_x'], row['label_y'], row['label'], crs=ccrs.GOOGLE_MERCATOR).opts(**_label_opts)
        for _, row in rel_unique.iterrows()
    ])

    ### compose map
    swap_map = tiles * base * patch_layer * acq_layer * rel_layer * acq_labels * rel_labels
    swap_map = swap_map.opts(title='Proposed Land Swaps — Ranked by Boundary Exposure Reduction')

    ### render to Bokeh, hv.render() overrides color= via its internal color cycle,
    ### so we patch fill colors directly on each glyph renderer afterward
    bokeh_fig = hv.render(swap_map)

    ### collect all MultiPolygon renderers in layer order
    ### order: base (alpha=0), patch, acq, rel
    _mp_rs = [r for r in bokeh_fig.renderers
              if isinstance(r, GlyphRenderer) and isinstance(r.glyph, _MP)]

    ### patch fill color and alpha on each named layer (skip index 0 = transparent base)
    _colors = [PATCH_COLOR, ACQUIRE_COLOR, RELEASE_COLOR]
    _alphas = [0.65, 0.9, 0.9]

    for _r, _c, _a in zip(_mp_rs[1:1 + len(_colors)], _colors, _alphas):
        _r.glyph.fill_color = _c
        _r.glyph.fill_alpha = _a

    ### add legend using zero-data dummy renderers so click-to-hide works per category
    _empty = ColumnDataSource({'xs': [[]], 'ys': [[]]})
    _r1 = bokeh_fig.patches('xs', 'ys', source=_empty, fill_color=PATCH_COLOR,   fill_alpha=0.65, line_color='black')
    _r2 = bokeh_fig.patches('xs', 'ys', source=_empty, fill_color=ACQUIRE_COLOR, fill_alpha=0.9,  line_color='black')
    _r3 = bokeh_fig.patches('xs', 'ys', source=_empty, fill_color=RELEASE_COLOR, fill_alpha=0.9,  line_color='black')

    legend_items = [
        LegendItem(label='Federal patch',                   renderers=[_r1]),
        LegendItem(label='Acquire (non-federal → federal)', renderers=[_r2]),
        LegendItem(label='Release (federal → non-federal)', renderers=[_r3]),
    ]

    bokeh_fig.add_layout(Legend(
        items=legend_items, location='top_right', click_policy='hide'
    ), 'right')

    ### display inline — components() reuses BokehJS already loaded by hv.extension
    _script, _div = components(bokeh_fig)
    display(HTML(_div + _script))

    ### save standalone HTML to figures directory
    map_out = os.path.join(figures_parcel_matrix_dir, 'land_swap_proposals_interactive.html')
    with open(map_out, 'w') as _f:
        _f.write(file_html(bokeh_fig, CDN, 'Land Swap Proposals'))
    print(f'Saved: {map_out}')

Saved: C:\Users\naho5798\Documents\Earth Data Cert\Final Project\Pawnee-Grasslands-Project\figures\parcel_matrix\land_swap_proposals_interactive.html
